# SVCM — Terminology Consumer — Cas de test nominal

**Profil** : IHE ITI SVCM
**Acteur testé** : SVCM-Terminology Consumer (`PAT_DEVICE_COLLECTEUR_ANALYSEUR_DE_DONNEES_CAD_CDM`)
**Répertoire cible** : SMT national ANS (`smt.esante.gouv.fr`)
**Référence test** : [SVCM_Cas de test nominal](https://interop.esante.gouv.fr/tm/testing/testsDefinition/viewTestPage.seam?id=12231&cid=29222)
**Référence FHIR server** : [Global features FHIR Server v5](https://esante.gouv.fr/sites/default/files/media/document/Global_features_FHIR_Server_version_finale_v5.pdf)

Ce notebook exécute l'intégralité des étapes du cas de test nominal et constitue la **trace + preuve** à soumettre au moniteur Gazelle.

## Configuration

In [ ]:
import requests
import json
import time
import os
from datetime import datetime
from urllib.parse import quote

FHIR_BASE = "https://smt.esante.gouv.fr/fhir"
API_BASE  = "https://smt.esante.gouv.fr/api"
WP_BASE   = "https://smt.esante.gouv.fr/wp-json/ans"
API_KEY = os.environ.get("SMT_API_KEY") or getpass.getpass("SMT API Key: ")

HTTP_RETRIES = 3
HTTP_BACKOFF = 2

HEADERS_JSON = {
    "Accept": "application/fhir+json",
    "User-Agent": "SVCM-Consumer-CAD-CDM",
}
HEADERS_API = {
    "accept": "*/*",
    "X-API-KEY": API_KEY,
    "User-Agent": "SVCM-Consumer-CAD-CDM",
}

# ── run output directory ──────────────────────────────────────────────────────
NOTEBOOK_ID = "01_nominal"
RUN_TS  = datetime.now().strftime("%Y%m%dT%H%M%S")
RUN_DIR = os.path.join("runs", f"{NOTEBOOK_ID}_{RUN_TS}")
os.makedirs(RUN_DIR, exist_ok=True)

# ── helpers ───────────────────────────────────────────────────────────────────

def http_get(url, headers=None):
    if headers is None:
        headers = HEADERS_JSON
    for attempt in range(1, HTTP_RETRIES + 1):
        try:
            print(f"  → GET {url}")
            r = requests.get(url, headers=headers, allow_redirects=True)
            if r.status_code == 200:
                return r
            raise Exception(f"HTTP {r.status_code}: {r.text[:200]}")
        except Exception as e:
            print(f"  ✗ tentative {attempt}/{HTTP_RETRIES} : {e}")
            if attempt == HTTP_RETRIES:
                raise
            time.sleep(HTTP_BACKOFF ** attempt)

def save_artifact(step, filename, data, binary=False):
    """Save a test artifact to the run directory, prefixed with the step number."""
    path = os.path.join(RUN_DIR, f"step{step:03d}_{filename}")
    if binary:
        with open(path, "wb") as f:
            f.write(data)
    else:
        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2, ensure_ascii=False)
    print(f"  ✓ {path}")
    return path

def print_keys(data, *keys):
    subset = {k: data.get(k) for k in keys if k in data}
    print(json.dumps(subset, indent=2, ensure_ascii=False))


def download_artifact(step, filename, url, headers=None):
    """Stream a large binary file to disk in chunks — avoids loading into memory."""
    if headers is None:
        headers = HEADERS_API
    path = os.path.join(RUN_DIR, f"step{step:03d}_{filename}")
    print(f"  → GET {url}")
    with requests.get(url, headers=headers, stream=True, allow_redirects=True) as r:
        if r.status_code != 200:
            raise Exception(f"HTTP {r.status_code}: {r.text[:200]}")
        total = int(r.headers.get("content-length", 0))
        downloaded = 0
        with open(path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):  # 1 MB chunks
                f.write(chunk)
                downloaded += len(chunk)
                if total:
                    print(f"\r  {downloaded/1024/1024:.1f} / {total/1024/1024:.1f} MB "
                          f"({downloaded/total*100:.0f}%)", end="", flush=True)
        if total:
            print()
    print(f"  ✓ {path}  ({downloaded/1024/1024:.1f} MB)")
    return path

print(f"Configuration OK — sortie dans : {RUN_DIR}")


---
## Étapes 10–30 — Récupération du CapabilityStatement général

**Transaction** : SVCM-ITI-95
**Requête** : `GET /fhir/metadata?_format=json`
**Objectif** : Vérifier que le SMT expose bien un CapabilityStatement FHIR valide et l'intégrer.

In [ ]:
# Étape 10 — Construction + envoi de la requête
# Étape 20 — Réception et intégration
# Étape 30 — PREUVE

url_cs = f"{FHIR_BASE}/metadata?_format=json"
r_cs = http_get(url_cs)
cs = r_cs.json()

save_artifact(30, "smt-cs.json", cs)

print("\n[PREUVE étape 30] Champs clés du CapabilityStatement :")
print_keys(cs, "resourceType", "status", "date", "kind", "fhirVersion", "software")

print(f"\nNombre de ressources supportées : {len(cs.get('rest', [{}])[0].get('resource', []))}")
resource_types = [r["type"] for r in cs.get("rest", [{}])[0].get("resource", [])]
print("Types :", resource_types)


---
## Étapes 50–70 — Récupération du CapabilityStatement terminologique

**Transaction** : SVCM-ITI-95
**Requête** : `GET /fhir/metadata?_format=json&mode=terminology`
**Objectif** : Vérifier les capacités spécifiquement terminologiques du SMT.

In [ ]:
# Étape 50 — Construction + envoi
# Étape 60 — Réception et intégration
# Étape 70 — PREUVE

url_tc = f"{FHIR_BASE}/metadata?_format=json&mode=terminology"
r_tc = http_get(url_tc)
tc = r_tc.json()

save_artifact(70, "smt-tc.json", tc)

print("\n[PREUVE étape 70] Champs clés du CapabilityStatement terminologique :")
print_keys(tc, "resourceType", "status", "date", "kind", "fhirVersion")

operations = [
    op.get("name")
    for res in tc.get("rest", [{}])[0].get("resource", [])
    for op in res.get("operation", [])
]
if operations:
    print("\nOpérations terminologiques disponibles :", sorted(set(operations)))


---
## Étapes 90–110 — Récupération d'une terminologie : TRE-R256-TypeMessagerie

**Transaction** : SVCM-ITI-95
**Requête** : `GET /fhir/CodeSystem?url=https://mos.esante.gouv.fr/NOS/TRE_R256-TypeMessagerie/FHIR/TRE-R256-TypeMessagerie&_format=json`
**Objectif** : Retrouver la terminologie par URL canonique, vérifier `content=complete`, récupérer son `id` et `versionId`.

In [ ]:
# Étape 90 — Construction + envoi
TRE256_URL = "https://mos.esante.gouv.fr/NOS/TRE_R256-TypeMessagerie/FHIR/TRE-R256-TypeMessagerie"
url_tre256 = f"{FHIR_BASE}/CodeSystem?url={quote(TRE256_URL)}&_format=json"

r_tre256 = http_get(url_tre256)
bundle_tre256 = r_tre256.json()

save_artifact(100, "smt-tre256-bundle.json", bundle_tre256)

# Étape 100 — Intégration
entries = bundle_tre256.get("entry", [])
assert len(entries) > 0, f"Aucune entrée trouvée pour {TRE256_URL}"
cs_entry   = entries[0]
cs_resource = cs_entry["resource"]
full_url   = cs_entry.get("fullUrl", "")

print(f"Entrées trouvées : {len(entries)}")
print(f"  content = {cs_resource.get('content')}  (attendu: complete)")
assert cs_resource.get("content") == "complete", "ERREUR : content != complete"

# Étape 110 — PREUVE : GET par fullUrl, parsing de id et versionId
print(f"\n[PREUVE étape 110] GET par fullUrl : {full_url}")
r_direct = http_get(f"{full_url}?_format=json")
cs_direct = r_direct.json()

cs_id      = cs_direct.get("id")
version_id = cs_direct.get("meta", {}).get("versionId")
version    = cs_direct.get("version")

print(f"  id        = {cs_id}")
print(f"  versionId = {version_id}")
print(f"  version   = {version}")
print(f"  name      = {cs_direct.get('name')}")
print(f"  concepts  = {len(cs_direct.get('concept', []))}")


---
## Étapes 130–150 — Historique et diff entre versions de TRE-R256

**Requêtes** :
- `GET /fhir/CodeSystem/{id}/_history?_format=json` → récupérer les entrées d'historique
- Comparer les deux dernières versions (concepts ajoutés / supprimés / libellés modifiés)

In [ ]:
# Étape 130 — Récupération de l'historique
url_hist = f"{FHIR_BASE}/CodeSystem/{cs_id}/_history?_format=json"
r_hist = http_get(url_hist)
hist = r_hist.json()

hist_entries = hist.get("entry", [])
total_versions = hist.get("total", len(hist_entries))
print(f"[PREUVE étape 130] Nombre total de versions : {total_versions}")
assert len(hist_entries) >= 2, "Moins de 2 versions disponibles, diff impossible"

# Les entrées sont triées du plus récent au plus ancien
latest_vid = hist_entries[0]["resource"]["meta"]["versionId"]
prev_vid   = hist_entries[1]["resource"]["meta"]["versionId"]

url_latest = f"{FHIR_BASE}/CodeSystem/{cs_id}/_history/{latest_vid}?_format=json"
url_prev   = f"{FHIR_BASE}/CodeSystem/{cs_id}/_history/{prev_vid}?_format=json"
print(f"Version courante   : _history/{latest_vid}")
print(f"Version précédente : _history/{prev_vid}")

cs_latest = http_get(url_latest).json()

# Étape 140 — Version précédente
cs_prev = http_get(url_prev).json()

print(f"\nVersion courante   : {cs_latest.get('version')} ({len(cs_latest.get('concept', []))} concepts)")
print(f"Version précédente : {cs_prev.get('version')} ({len(cs_prev.get('concept', []))} concepts)")


In [ ]:
# Étape 150 — PREUVE : diff entre les deux versions

def extract_concepts_recursive(concepts, prefix=""):
    flat = {}
    for c in concepts:
        code_val = c.get("code")
        flat[code_val] = {"display": c.get("display"), "path": prefix + "/" + code_val}
        if "concept" in c:
            flat.update(extract_concepts_recursive(c["concept"], prefix + "/" + code_val))
    return flat

c_prev   = extract_concepts_recursive(cs_prev.get("concept", []))
c_latest = extract_concepts_recursive(cs_latest.get("concept", []))

added   = set(c_latest) - set(c_prev)
removed = set(c_prev) - set(c_latest)
changed = {k for k in set(c_prev) & set(c_latest) if c_prev[k]["display"] != c_latest[k]["display"]}

diff_result = {
    "from_version": cs_prev.get("version"),
    "to_version":   cs_latest.get("version"),
    "added":   sorted(added),
    "removed": sorted(removed),
    "display_changed": {k: {"before": c_prev[k]["display"], "after": c_latest[k]["display"]}
                        for k in sorted(changed)},
}
save_artifact(150, "diff-tre256.json", diff_result)

print(f"[PREUVE étape 150] Diff {cs_prev.get('version')} → {cs_latest.get('version')}")
print(f"  Codes ajoutés    : {len(added)}")
print(f"  Codes supprimés  : {len(removed)}")
print(f"  Libellés modifiés: {len(changed)}")
if not added and not removed and not changed:
    print("  Aucune différence entre les deux versions.")


---
## Étapes 170–190 — Récupération de SNOMED CT International

**Note** : `content = "not-present"` est attendu (SNOMED n'est pas hébergé inline dans le FHIR server).

In [ ]:
# Étape 170 — Métadonnées FHIR SNOMED CT International
url_snomed = f"{FHIR_BASE}/CodeSystem?_format=json&url=http://snomed.info/sct"
r_snomed = http_get(url_snomed)
bundle_snomed = r_snomed.json()

save_artifact(170, "snomed-bundle.json", bundle_snomed)

print("[PREUVE étape 170] Entrées SNOMED CT :")
for e in bundle_snomed.get("entry", []):
    res = e.get("resource", {})
    print(f"  fullUrl : {e.get('fullUrl')}")
    print(f"  title   : {res.get('title')}")
    print(f"  version : {res.get('version')}")
    print(f"  content : {res.get('content')}  (attendu: not-present)")
    print()


In [ ]:
# Étape 180 — Fiche détaillée SNOMED CT International (FHIR public)
url_snomed_detail = f"{FHIR_BASE}/CodeSystem/900000000000207008-20260301?_format=json"
r_detail = http_get(url_snomed_detail)
snomed_detail = r_detail.json()
save_artifact(180, "snomed-detail.json", snomed_detail)
print("Détail CodeSystem SNOMED international :")
print_keys(snomed_detail, "id", "version", "name", "title", "status", "content")


In [ ]:
# Étape 180 (suite) — Liste et versions via API propriétaire (nécessite API_KEY)
print("Liste des terminologies (API) :")
r_list = http_get(f"{API_BASE}/terminologies/list", HEADERS_API)
terminologies = r_list.json()
snomed_entries = [t for t in (terminologies if isinstance(terminologies, list) else [])
                  if "snomed" in str(t).lower()]
print(f"  Entrées SNOMED dans la liste : {len(snomed_entries)}")

print("\nVersions détaillées SNOMED CT :")
url_versions = f"{WP_BASE}/terminologies/versions-details?terminologyId=terminologie-snomed-ct"
r_versions = http_get(url_versions, HEADERS_API)
versions_data = r_versions.json()
save_artifact(180, "snomed-versions.json", versions_data)
print(json.dumps(versions_data, indent=2, ensure_ascii=False)[:800])


In [ ]:
# Étape 190 — PREUVE : téléchargement SNOMED CT zip (streaming)
snomed_zip_url = (
    f"{WP_BASE}/terminologies/zip"
    "?terminologyId=terminologie-snomed-ct"
    "&version=Janvier%202026%20v1.0"
    "&licenceConsent=true&dataTransferConsent=true"
)
download_artifact(190, "snomed-ct.zip", snomed_zip_url)


---
## Étapes 210–230 — Récupération de SNOMED CT France (Extension FR)

In [ ]:
# Étape 210 — Fiche détaillée SNOMED CT FR
url_snomed_fr = f"{FHIR_BASE}/CodeSystem/11000315107-20250621?_format=json"
r_snomed_fr = http_get(url_snomed_fr)
snomed_fr = r_snomed_fr.json()
save_artifact(210, "snomed-fr-detail.json", snomed_fr)

print("[PREUVE étape 210] Détail CodeSystem SNOMED CT FR :")
print_keys(snomed_fr, "id", "version", "name", "title", "status", "content")

# Étape 220 — Versions détaillées SNOMED CT FR (nécessite API_KEY)
url_versions_fr = f"{WP_BASE}/terminologies/versions-details?terminologyId=terminologie-snomed-ct-fr"
r_versions_fr = http_get(url_versions_fr, HEADERS_API)
versions_fr = r_versions_fr.json()
save_artifact(220, "snomed-fr-versions.json", versions_fr)
print("\nVersions :")
print(json.dumps(versions_fr, indent=2, ensure_ascii=False)[:800])


In [ ]:
# Étape 230 — PREUVE : téléchargement SNOMED CT FR zip (streaming)
snomed_fr_zip_url = (
    f"{WP_BASE}/terminologies/zip"
    "?terminologyId=terminologie-snomed-ct-fr"
    "&version=Juin%202025%20v1.0"
    "&licenceConsent=true&dataTransferConsent=true"
)
download_artifact(230, "snomed-ct-fr.zip", snomed_fr_zip_url)


---
## Récapitulatif

In [ ]:
print(f"Run : {RUN_DIR}")
print(f"{'Fichier':<45} {'Taille':>10}")
print("-" * 57)
for fname in sorted(os.listdir(RUN_DIR)):
    fpath = os.path.join(RUN_DIR, fname)
    size  = os.path.getsize(fpath)
    size_str = f"{size/1024:.1f} KB" if size < 1_000_000 else f"{size/1024/1024:.1f} MB"
    print(f"  {fname:<43} {size_str:>10}")
print("\n✓ Cas de test nominal SVCM-Terminology-Consumer terminé.")
